# Machine Learning data compression lab session – Data Transmission and Cryptography

In this lab session, a small demonstration of `tensorflow` will be conducted, illustrating some of the practical challenges of training and using neural networks for data compression. After getting some practice, you'll be walked introduced to the widely refenced Ballé *et al.* 2017 and 2018 architectures for image compression and challenged to implement variants of them.

Ready? Let's go!

## How to install `tensorflow`?

In this session we'll use the `tensorflow` Python library and other derivative libraries. The following code blocks show how to install it and import it for usage.

In [ ]:
# pip install tensorflow
import tensorflow as tf

A *tensor* is a structured array; informally we could say a "vector with shape". A vector is, in particular, a 1-dimensional tensor, and a matrix is a 2-dimensional tensor. Colour images can be understood as a 3-dimensional tensors: three 2-dimensional arrays, each corresponding to one of the colour channels Red, Green or Blue (RGB). Tensors constitute the core of `tensorflow` and are the main object we'll be manipulating.

A tensor object has a fixed number of dimensions and size (its shape), and a fixed type for all its entries (such as strings, 32-bit floating point numbers, or 16-bit unsigned integers). Tensors with different shapes or data types may not be operated together: adding, comparing, etc.

In [ ]:
x = tf.constant([[1,0],[1,1]], dtype=tf.float32)
y = tf.constant([[1,0],[2,1]], dtype=tf.int32)
z = tf.constant([[1,0]], dtype=tf.float32)

print(x)
print(x+y)
print(x+z)

## Neural networks with `tensorflow`

In this section we'll construct a demonstration neural network using tensorflow. First, however, we need to generate some training data!



### An artificial training data set

We'll construct a random dataset of 200 10-dimensional vectors (that is, tensors with shape (10,), $x \in \mathbb{R}^{10}$) and a set of 4-dimensional tags for those vectors, $y \in \mathbb{R}^4$, that will be a fixed linear combination of those vectors $y=Ax : A \in \text{Mat}_{4\times 10}(\mathbb{R})$. Our neural network will have to learn a function that maps $x$ to $y$; if successful, it should obtain parameters close to the entries of $A$.

Observe that uniformly distributed random samples are drawn from the $(0,1)$ interval.

In [ ]:
A = tf.random.uniform((4,10))
X = tf.random.uniform((200,10))
Y = tf.transpose(tf.linalg.matmul(A,X, transpose_b=True), (1,0))

print(A)
print(X.shape)
print(Y.shape)

### A simple toy model

Having generated our training data `X,Y`, we will construct a 1-layer sequential model to approximate $A$. Our model will be, in essence, a matrix like the one we want to approximate, and we'll use stochastic gradient descent (SDG) to carry out said approximation.

In [ ]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(4, activation=None, use_bias=True))
model.compile(loss='mse')
model.fit(x=X, y=Y, batch_size=20, epochs=100, verbose=True)

Having trained the model, how good is it as an approximation of $A$? Let's find out by evaluating it in some vectors:

In [ ]:
i = 0
print(model(X[i:i+1,:]))
print(Y[i,:])

We may also define an alternative loss function of our choosing, such as mena absolute error:

In [ ]:
# Custom loss function
def custom_loss(y_true, y_pred):
    return tf.reduce_mean(abs(y_true-y_pred))

# Model definition and training
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(4, activation=None, use_bias=True))
model.compile(loss=custom_loss)
model.fit(x=X, y=Y, batch_size=20, epochs=100, verbose=True)

# Model evaluation
i = 0
print(model(X[i:i+1,:]))
print(Y[i,:])

Not bad! Yet, so many questions!

*   Would results be better if our model were trained for longer?
*   What if we changed the batchsize?
*   What if we use a different customised loss function?
*   What if we don't use a bias vector? Would it still work?
*   What if we set a non-linear activation such as ReLU?
*   Using Adam as the optimiser, do we improve performance when training for the same amount of time?

You may evaluate some of these in the following code cell. Play around!

### Exercise 1 – Stability of equilibria

Build a linear autoencoder for an artificial dataset:
1. Generate a uniformly random set of 4-dimensional vectors: $y$.
2. With a randomly-generated matrix $A \in \text{Mat}_{100 \times 4}(\mathbb{R})$, generate a linearly-correlated set of 100-dimensional vectors, $x$.
3. Create a sequential neural network composed of two layers (or blocks of layers): one (the *encoder*) that maps your 100-dimensional vectors to 4-dimensional vectors, and another (the *decoder*) that maps them back to 100-dimensional vectors.
4. Train that network for a loss function such as $\operatorname{MSE}$. Does the network manage to recover the vectors $x$?
5. Now try the same *without* dimensionality reduction; mapping the data into 100-dimensional vectors at all layers. Does the network manage to recover the vectors $x$?

In [ ]:
# Data definition
A = tf.random.uniform((100,4))
seed = tf.random.uniform((200,4))
data = tf.transpose(tf.linalg.matmul(A, seed, transpose_b=True), (1,0))

# Model definition and training
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(4, activation=None, use_bias=False))
model.add(tf.keras.layers.Dense(100, activation=None, use_bias=False))
model.compile(loss='mse')
model.fit(x=data, y=data, batch_size=20, epochs=500, verbose=True)

# Model evaluation
i = 0
print(model(100*data[i:i+1,:]))
print(100*data[i:i+1,:])

### Exercise 2 – Neural network architecture choices

In the Campus Virtual you'll be provided with the FIFA World Cup matches data set. You may upload it to this notebook for the purposes of the following exercise. To manipulate the non-quantitative parts of the data (dates, country names, etc.), consider converting them to numerical values that a neural network can actually process.

As part of this exercise, you'll aim to **build a neural network that can predict whom will win a given match**, given the date/year of the match and the countries playing as input.

First, you'll need to make decisions regarding the architecture of your network.

1.   What are the network inputs? What is their size and type?
2.   What will be the final output of the network? Will it be a chance of winning? An array of probabilities? A prediction of the number of goals by each team?
3.   How many layers will your network have? Of what type? Of what size? What will your activation choices be?
4.   What is the optimisation goal? What will be the loss function to achieve it?

Test your network. Is it accurate in predicting past results?

In [ ]:
# Basis code paraphrased from previous code cell.
# Data definition
import numpy as np
data = np.genfromtxt('TDS2526-WorldCupDataset.csv', delimiter=',', dtype=str) # This loads the contents of the CSV as an array of strings.
# You'll have to first manipulate the table into two variables, 'train_data' and 'goal_data'.

# Model definition
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(100, activation=None, use_bias=False))
# Build whatever architecture you see fit.
model.compile(loss='mse')

# Model training
model.fit(x=train_data, y=goal_data, batch_size=10, epochs=500, verbose=True)

# Model evaluation on a particular datapoint
i = 0
print(model(data[i:i+1,:]))
print(data[i:i+1,:])

## Neural network prediction

Open an image of your choice as a 3-dimensional array using Tensorflow. In this section you'll develop a predictor-based codec.

First, we need to set up our training data. We'll use a neighbourhood of the previously visited pixels to predict the following pixel value. For example, the following code cell generates train data from the previous single pixel value to predict the single next pixel value.

### Exercise

Modify the following cell so that the training data vectors are the top, left, and previous band values (or 0, if not available) for each pixel value.

In [ ]:
filename = "image.raw"
bands = 1
height = 512
width = 512

# Raw image opening

string = tf.io.read_file(filename)
image = tf.cast(tf.io.decode_raw(string, datatype, little_endian=(endianess==1)), tf.float32)

# Image file opening

img = tf.keras.utils.load_img(filename, target_size=image_size)
image = tf.keras.utils.img_to_array(img)
image = tf.reshape(image, (-1))

train_data = []
goal_data = []

for i in range(tf.size(image)-1):
    train_data.append(image[i])
    goal_data.append(image[i+1])

train_data = tf.convert_to_tensor(train_data, dtype=tf.float32)
goal_data = tf.convert_to_tensor(goal_data, dtype=tf.float32)

Once we have our training data, we may use a simple network like those we used in previous exercises to predict the next pixel values. The following cell defines a simple linear network.

### Exercises

1. Train a network and calculate the entropy of the rounded prediction residuals. Is it better than the entropy of the original data?
2. Train a network for larger prediction contexts such as that from the previous exercise. Is this prediction better than the previous one in terms of entropy?
3. Modify the model's architecture to a non-linear network (for example, adding more layers and a non-linearity). Is the model now more accurate?
4. Modify the model's loss function into cross-entropy. Recall that the output of your model should have as many possible values as there are prediction values, and you should use a softmax function to normalise it into a probability distribution. Is this better for prediction in your case?

In [ ]:
# Custom loss function
def custom_loss(x_true, x_pred, y):
    return tf.reduce_mean((x_true-x_pred)**2) + tf.reduce_mean(y**2)

# Model definition
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(1, activation=None, use_bias=False))
model.compile(loss=custom_loss)

# Model training
model.fit(x=train_data, y=goal_data, batch_size=20, epochs=100, verbose=True)

# Model evaluation
i = 100
print(model(train_data[i]))
print(goal_data[i])

residuals = tf.round(model(train_data)-goal_data)
print(residuals[i])

## Compression autoencoders

As we've seen in the lectures, the state of the art in (lossy) image compression are autoencoders. Unlike the example in exercise 1, these are not just trained to minimise the distortion between the inputs and outputs, but to also minimise the coding rate of the latent representation, a rate-distortion minimisation problem: $$L = R(\hat{y}) + \lambda D(x, hat{x}).$$

As we've seen in the lectures, the key innovation is in the definition of the rate term of that loss function, since the entropy of a vector is not a differentiable function. What we do instead is to *fit* the latent representation to a pre-defined probability distribution, that is, we try to make the distribution of the latent values resemble as much as possible a pre-defined shape. From another perspective, what we'll do is train our model so that the latent representation (our *compressed* tensor) can be optimally coded using that pre-fixed probability distribution.

Consider the probability density function of our prior, $p(x)$. The goal is to minimise the expected codeword length of each latent value $y_i$. You may recall from previous sessions that the expected codeword length of a symbol with probability $p$ is $-\log(p)$, thus our goal here will be to minimise $-\log\left(p(y_i)\right)$ across the latent representation, or more precisely to minimise $$R(y) = \sum_{i} -\log\left(p(y_i)\right)$$.

For this example, we will consider our prior distribution to be a standard Gaussian distribution, $N(0,1)$. The probability density function of this distribution is $p(x) = \frac{1}{\sqrt{2\pi}} e^{\frac{-x^2}{2}}$. With this, the rate term of the loss function will be equivalent to $$R(y) = \sum_{i} -\log\left(p(y_i)\right) = \sum_{i} y_i^2.$$

### Exercise

Prove that the previous derivation of the rate term is indeed true.

Using this fixed standard Gaussian distribution, we will build a small autoencoder to compress some simple information vectors. Consider the following set of artificial data points:

In [ ]:
seed = 4*tf.random.normal((200,1))
noise = 0.5*tf.random.uniform((200,4))

# From a single real seed from U(0,4), we create 4-dimensional data vectors using different functions for each component.
# Furthermore, we add uniform [-0.5, 0.5] noise to our data.
data = tf.transpose(tf.squeeze(tf.convert_to_tensor([2*seed, 3*seed -1, seed**2, seed**2-2*seed+1])), (1,0))+noise

We can now define an autoencoder to try to encode this data efficiently using the proposed rate-distortion loss function. In the following cell you'll find a linear autoencoder setup.

In [ ]:
# Model definition and training
class EncoderTransform(tf.keras.Sequential):
    """The encoder transform."""
    def __init__(self):
        super().__init__(name="encoder")
        self.add(tf.keras.layers.Dense(1, activation=None, use_bias=False))

class DecoderTransform(tf.keras.Sequential):
      """The decoder transform."""
      def __init__(self):
          super().__init__(name="decoder")
          self.add(tf.keras.layers.Dense(4, activation=None, use_bias=False))

class Autoencoder(tf.keras.Model):
    """The encoder transform."""
    def __init__(self, lmbda):
        super().__init__()
        self.encoder = EncoderTransform()
        self.decoder = DecoderTransform()
        self.lmbda = lmbda

    def call(self, x):
        """Computes rate and distortion loss."""
        y = self.encoder(x)
        x_hat = self.decoder(y)

        rate_loss = tf.reduce_mean(y**2)
        distortion_loss = tf.reduce_mean((x-x_hat)**2)
        return rate_loss + self.lmbda*distortion_loss

    def compress(self, x):
        return self.encoder(x)

    def decompress(self, y):
        return self.decoder(y)

model = Autoencoder(1) # Instantiate a model with lambda=1.
model.compile(loss='mse')
model.fit(x=data, y=data*0, batch_size=10, epochs=500, verbose=True) # Target data is set to 0 to ensure that we optimise our model to minimise the loss function directly.


# Model evaluation
i = 0
print('Latent representation:')
print(model.compress(data[i:i+1,:]))
print('Reconstruction:')
print(model.decompress(model.compress(data[i:i+1,:])))
print('Actual value:')
print(data[i:i+1,:])

That autoencoder was probably not very accurate. Here are some issues we can observe:

1. The architecture cannot replicate the type of correlation we want to exploit in our data. Our transforms are linear products, but some of the relations in our data are non-linear (polynomials, to be exact).
2. You may notice the latent representation value was very small. This is because by minimising $\sum_i y^2$ we incentivise those values to be as small as possible. If we were to quantise the latent representation (for example, rounding to integers) to encode it, we would probably round them all to 0 and lose all information.

We will fix those issues in the following exercise.

### Exercise

- To avoid the vanishing latent problem, we will introduce additive uniform noise $U\left(-\frac{1}{2}, \frac{1}{2}\right)$ to our latent. Modify the autoencoder so that it maps an input vector $x$ to a latent $y$, adds the randomly generated uniform noise to the latent obtaining $y_hat$, and then calculates the reconstruction $x_hat$ and the loss function $L = R(\hat{y}) + \lambda D(x, \hat{x})$ using that noisy latent. Train the model again. What changes do you observe?
- To better represent the data, modify the encoder and decoder transforms to be non-linear transforms. These should have the capacity of producing the polynomial correlation we have in our data. What architecture could you use to achieve that? Train the model again. What changes do you observe.
- The $\lambda$ parameter regulates the trade-off between the rate and distortion in our loss function. Try training models with different $\lambda$ values and calculate the entropy of the quantised latents and the squared error of the reconstructions. Plot these results in a 2D graph (a rate-distortion plot). What do you observe?

## The Ballé 2017 architecture

We can download that script from the [Tensorflow Compression GitHub repository](https://github.com/tensorflow/compression). As training data, we'll use the [Kodak data set](https://www.kaggle.com/datasets/sherylmehta/kodak-dataset). Download both and upload them into this Colab notebook. In the lecture, the script will be described in detail.

To train an instance of that architecture, we'll use the following command:

In [ ]:
!python3 bls2017.py -V --model_path ./test_model train --lambda 0.01 --train_glob "./Kodak/*png" --num_filters 128 --epochs 10 --steps_per_epoch 10 --batchsize 2

### Exercises

1. Replace the layers in the main transform with residual blocks: two 3x3 convolutions, one with ReLU and one without activation (no downsampling), followed by the residual, and then (if we want downsampling) maxpooling and a ReLU.
2. Implement an attention module and insert it between layers 2 and 3: two 3x3 convolutions, one with ReLU and one with a sigmoid (no downsampling), followed by the scaling.

## The Ballé 2018 architecture

We can again download that script from the [Tensorflow Compression GitHub repository](https://github.com/tensorflow/compression). As training data, we'll  again use the [Kodak data set](https://www.kaggle.com/datasets/sherylmehta/kodak-dataset). Download both and upload them into this Colab notebook. In the lecture, the script will be described in detail.

To train an instance of that architecture, we'll use the following command:

In [ ]:
!python3 bmshj2018.py -V --model_path ./test_model train --lambda 0.01 --train_glob "./Kodak/*png" --num_filters 128 --epochs 10 --steps_per_epoch 10 --batchsize 2

### Exercises

1. Implement another hierarchical hyperprior to encode the hyperprior's side information. Train (for a short period) an instance of each architecture. Does it seem to improve performance?
2. The current hyperprior is just a scale hyperprior. Can you make it a mean and scale hyperprior? How about a Gaussian Mixture? The [Tensorflow documentation](https://www.tensorflow.org/api_docs/python/tfc/entropy_models/LocationScaleIndexedEntropyModel) may help.